# 🎤 Voice Service: Local Whisper (GPU) + gTTS

This notebook runs a FastAPI microservice for:
- **STT (Speech-to-Text)**: Local OpenAI Whisper model (runs on GPU, no API calls)
- **TTS (Text-to-Speech)**: gTTS (Google Text-to-Speech)
- **S2S (Speech-to-Speech)**: Full pipeline with optional AI integration

## Setup Steps:
1. Run cells in order (1-5)
2. Set your ngrok auth token only
3. Service starts on public ngrok URL
4. Copy URL to your backend `VOICE_REMOTE_URL` environment variable

✅ **No OpenAI API key required** - Uses local Whisper model on GPU

## Cell 1: Install Dependencies

⚠️ **Note**: First install may take 3-5 minutes

In [ ]:
import subprocess
import sys
import os

print("📦 Installing system dependencies (apt-get)...")
os.system('apt-get update -qq')
os.system('apt-get install -qq -y espeak-ng ffmpeg libsndfile1 > /dev/null 2>&1')
print("✅ System dependencies installed")

packages = ['fastapi', 'uvicorn', 'pyngrok']

print("\n📦 Installing Python packages...")
for pkg in packages:
    print(f"  Installing {pkg}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("\n📦 Installing OpenAI Whisper (local GPU inference)...")
try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'openai-whisper', '-q'],
        capture_output=True,
        text=True,
        timeout=120
    )
    if result.returncode == 0:
        print("✅ Whisper installed")
    else:
        print(f"⚠️  Trying alternative install...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/openai/whisper.git', '-q'], timeout=120)
        print("✅ Whisper installed from git")
except Exception as e:
    print(f"❌ Whisper installation failed: {str(e)[:100]}")
    raise

print("\n📦 Installing gTTS for Text-to-Speech...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gtts', '-q'], timeout=60)
print("✅ gTTS installed")

print("\n📦 Installing optional speedup (faster-whisper)...")
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faster-whisper', '-q'], timeout=120)
    print("✅ faster-whisper installed (optional speedup enabled)")
except:
    print("⚠️  faster-whisper skipped (optional)")

print("\n📦 Note: Skipping Coqui XTTS (using gTTS instead)...")
print("\n✅ All dependencies installed successfully!")
print("\n📊 Installed components:")
print("   ✅ FastAPI + Uvicorn")
print("   ✅ ngrok (for public URL)")
print("   ✅ Local Whisper (GPU STT)")
print("   ✅ gTTS (reliable TTS)")
print("   ✅ System dependencies (ffmpeg, etc)")
print("\n🎯 Ready: No OpenAI API key needed - all processing is local!")

📦 Installing system dependencies (apt-get)...
✅ System dependencies installed

📦 Installing Python packages...
  Installing fastapi...
  Installing uvicorn...
  Installing pyngrok...

📦 Installing OpenAI Whisper (local GPU inference)...
✅ Whisper installed

📦 Installing gTTS for Text-to-Speech...
✅ gTTS installed

📦 Installing optional speedup (faster-whisper)...
✅ faster-whisper installed (optional speedup enabled)

📦 Note: Skipping Coqui XTTS (using gTTS instead)...

✅ All dependencies installed successfully!

📊 Installed components:
   ✅ FastAPI + Uvicorn
   ✅ ngrok (for public URL)
   ✅ Local Whisper (GPU STT)
   ✅ gTTS (reliable TTS)
   ✅ System dependencies (ffmpeg, etc)

🎯 Ready: No OpenAI API key needed - all processing is local!


## Cell 2: Set Environment Variables

⚠️ **IMPORTANT**:
1. Get your ngrok auth token from https://dashboard.ngrok.com/auth/your-authtoken (FREE account)
   - Sign up at https://dashboard.ngrok.com/signup if you don't have an account
   - ngrok token is required for public URL
2. OpenAI API key NOT needed - uses local Whisper on GPU

In [ ]:
import os

# 🔑 ngrok auth token (REQUIRED for public URL - get from https://dashboard.ngrok.com/auth/your-authtoken)
# Free ngrok account at: https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = "35yLZfOFH6h2J1mG2vcCD8kaC5C_5VQR8peRrSVa95SBDKtVi"  # ⚠️ REPLACE THIS
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN

# Optional: AI endpoint for S2S mode
AI_REPLY_URL = "http://localhost:8000/api/v1/chat/message"
ENABLE_AI_REPLY = "false"  # Set to "true" to enable AI integration

# Whisper model size (base, small, medium, large)
WHISPER_MODEL_SIZE = "base"  # Change to small/medium/large for better accuracy

# Validate credentials
if not NGROK_AUTH_TOKEN or NGROK_AUTH_TOKEN == "your-ngrok-token-here":
    print("❌ ERROR: ngrok auth token not set!")
    print("   1. Sign up (free): https://dashboard.ngrok.com/signup")
    print("   2. Get token: https://dashboard.ngrok.com/auth/your-authtoken")
    print("   3. Paste token in NGROK_AUTH_TOKEN above")
    raise ValueError("NGROK_AUTH_TOKEN is required")

print("✅ Environment variables set")
print(f"   Whisper Model: {WHISPER_MODEL_SIZE}")
print(f"   ngrok Auth Token: {NGROK_AUTH_TOKEN[:10]}...")
print(f"   AI Integration: {ENABLE_AI_REPLY}")
print("   OpenAI API Key: NOT NEEDED - Local Whisper on GPU")
print("\n✅ Ready to proceed to Cell 3!")

✅ Environment variables set
   Whisper Model: base
   ngrok Auth Token: 35yLZfOFH6...
   AI Integration: false
   OpenAI API Key: NOT NEEDED - Local Whisper on GPU

✅ Ready to proceed to Cell 3!


## Cell 3: Setup pyngrok for Public URL

In [ ]:
from pyngrok import ngrok
import json

# Optional: set auth token for stable URLs
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok auth token set")
else:
    print("⚠️  ngrok will use free tier (URL changes on restart)")

print("\n✅ pyngrok ready")

✅ ngrok auth token set

✅ pyngrok ready


## Cell 4: Define FastAPI Application

This cell defines all endpoints using LOCAL Whisper on GPU:
- `GET /` - Root
- `GET /health` - Health check
- `POST /transcribe` - Audio → Text (Local Whisper GPU)
- `POST /synthesize` - Text → Audio (gTTS)
- `POST /s2s` - Audio → Text (Local Whisper) → AI Reply → Audio (gTTS)

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
import whisper
import io
import base64
import tempfile
import requests
from typing import Optional
from gtts import gTTS
import os

# Initialize app
app = FastAPI(title="Voice Service", version="1.0.0")

# Global state: Load Whisper model on GPU
print(f"🚀 Loading Whisper model ({WHISPER_MODEL_SIZE}) on GPU...")
whisper_model = whisper.load_model(WHISPER_MODEL_SIZE)
print(f"✅ Whisper model loaded: {WHISPER_MODEL_SIZE}")

WHISPER_AVAILABLE = True
TTS_AVAILABLE = True  # Using gTTS - always available

class SynthesizeRequest(BaseModel):
    text: str
    language: str = "en"

@app.get("/")
async def root():
    """Root endpoint."""
    return {
        "service": "Voice STT/TTS Pipeline",
        "stt_engine": "Local Whisper (GPU)",
        "tts_engine": "gTTS (Google Text-to-Speech)",
        "endpoints": ["/transcribe", "/synthesize", "/s2s", "/health"],
        "docs": "/docs"
    }

@app.get("/health")
async def health():
    """Health check endpoint."""
    return {
        "status": "healthy",
        "whisper_available": WHISPER_AVAILABLE,
        "tts_available": TTS_AVAILABLE,
        "stt_engine": "Local Whisper",
        "tts_engine": "gTTS"
    }

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    """
    Transcribe audio file using local Whisper model on GPU.
    No OpenAI API key required - runs completely offline.
    """
    try:
        content = await file.read()

        # Save audio to temp file
        with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as tmp:
            tmp.write(content)
            temp_path = tmp.name

        try:
            print(f"📝 Transcribing {len(content)} bytes with local Whisper...")
            # Use local Whisper model
            result = whisper_model.transcribe(temp_path, language="en")
            text = result["text"]
            print(f"✅ Transcribed: {text[:50]}...")
            return {
                "text": text,
                "source": "local_opensource_whisper"
            }
        finally:
            os.unlink(temp_path)

    except Exception as e:
        print(f"❌ Transcription error: {e}")
        raise HTTPException(status_code=500, detail=f"Transcription error: {str(e)}")

@app.post("/synthesize")
async def synthesize(req: SynthesizeRequest):
    """
    Synthesize text to speech using gTTS (Google Text-to-Speech).
    No system dependencies, reliable, works on any platform.
    """
    try:
        print(f"🎵 Synthesizing: {req.text[:50]}...")

        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
            mp3_path = tmp.name

        # Generate speech with gTTS
        tts = gTTS(text=req.text, lang=req.language, slow=False)
        tts.save(mp3_path)

        print(f"✅ Synthesis complete")

        with open(mp3_path, "rb") as f:
            audio_bytes = f.read()

        os.unlink(mp3_path)

        audio_base64 = base64.b64encode(audio_bytes).decode("utf-8")

        return {
            "audio_url": f"data:audio/mp3;base64,{audio_base64}",
            "duration_ms": int(len(audio_bytes) / 16),
            "format": "audio/mp3",
            "engine": "gTTS"
        }

    except Exception as e:
        print(f"❌ Synthesis error: {e}")
        raise HTTPException(status_code=500, detail=f"Synthesis error: {str(e)}")

@app.post("/s2s")
async def s2s(file: UploadFile = File(...)):
    """
    Full speech-to-speech: STT (Local Whisper) → AI Reply → TTS (gTTS).
    """
    try:
        content = await file.read()

        # Step 1: Transcribe with local Whisper
        with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as tmp:
            tmp.write(content)
            temp_path = tmp.name

        try:
            print("📝 Step 1: Transcribing audio with local Whisper...")
            result = whisper_model.transcribe(temp_path, language="en")
            user_text = result["text"]
            print(f"   User said: {user_text}")
        finally:
            os.unlink(temp_path)

        # Step 2: Get AI reply (optional)
        ai_reply = user_text  # Default: echo

        if ENABLE_AI_REPLY == "true":
            print("🤖 Step 2: Getting AI reply...")
            try:
                response = requests.post(
                    AI_REPLY_URL,
                    json={"message": user_text, "session_id": "voice_s2s"},
                    timeout=120
                )
                if response.status_code == 200:
                    body = response.json()
                    ai_reply = body.get("reply") or body.get("text") or user_text
                    print(f"   AI said: {ai_reply}")
            except Exception as e:
                print(f"   ⚠️  AI endpoint error, using echo: {e}")

        # Step 3: Synthesize reply with gTTS
        print("🎵 Step 3: Synthesizing reply...")
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
            mp3_path = tmp.name

        tts = gTTS(text=ai_reply, lang="en", slow=False)
        tts.save(mp3_path)

        with open(mp3_path, "rb") as f:
            audio_bytes = f.read()

        os.unlink(mp3_path)

        audio_base64 = base64.b64encode(audio_bytes).decode("utf-8")

        print("✅ S2S pipeline complete")

        return {
            "request_text": user_text,
            "reply_text": ai_reply,
            "audio_url": f"data:audio/mp3;base64,{audio_base64}",
            "duration_ms": int(len(audio_bytes) / 16),
            "stt_engine": "local_whisper",
            "tts_engine": "gTTS"
        }

    except HTTPException:
        raise
    except Exception as e:
        print(f"❌ S2S error: {e}")
        raise HTTPException(status_code=500, detail=f"S2S error: {str(e)}")

print("✅ FastAPI app initialized")
print("   STT Engine: Local Whisper (GPU) - No API key needed!")
print("   TTS Engine: gTTS (Google Text-to-Speech)")
print("   All endpoints ready!")

🚀 Loading Whisper model (base) on GPU...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 164MiB/s]


✅ Whisper model loaded: base
✅ FastAPI app initialized
   STT Engine: Local Whisper (GPU) - No API key needed!
   TTS Engine: gTTS (Google Text-to-Speech)
   All endpoints ready!


## Cell 5: Start Service with Public ngrok URL

⚠️ **Important:**
- This cell will run the service indefinitely
- Copy the **public ngrok URL** and set it as `VOICE_REMOTE_URL` in your backend
- To stop, click the ⏹️ stop button or interrupt the cell

In [ ]:
from pyngrok import ngrok
from uvicorn import Server, Config
import asyncio

# Set ngrok auth token before connecting
print("🌐 Setting up ngrok tunnel...")
try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok auth token set")
except Exception as e:
    print(f"❌ ngrok auth failed: {e}")
    print("\n⚠️  ERROR: Could not authenticate with ngrok!")
    print("\n📋 Steps to fix:")
    print("   1. Sign up (free): https://dashboard.ngrok.com/signup")
    print("   2. Get auth token: https://dashboard.ngrok.com/auth/your-authtoken")
    print("   3. Update NGROK_AUTH_TOKEN in Cell 2")
    print("   4. Re-run Cell 2, then re-run Cell 5")
    raise

# Start ngrok tunnel
print("\n🌐 Starting ngrok tunnel...")
try:
    ngrok_tunnel = ngrok.connect(8001, "http")
    public_url = ngrok_tunnel.public_url
    print(f"✅ Public URL: {public_url}")
except Exception as e:
    print(f"❌ ngrok connection failed: {e}")
    print("\nMake sure NGROK_AUTH_TOKEN is set correctly in Cell 2")
    raise

print(f"\n📋 Copy this URL to your backend:")
print(f"   VOICE_REMOTE_URL={public_url}")

# Create Uvicorn config
config = Config(app, host="127.0.0.1", port=8001, log_level="info")
server = Server(config)

# Run server
print(f"\n🚀 Voice Service starting on {public_url}")
print(f"   Local: http://127.0.0.1:8001")
print(f"   Docs: {public_url}/docs")
print("\n⏹️  Press stop button to halt the service")
print("="*60)
await server.serve()

🌐 Setting up ngrok tunnel...
✅ ngrok auth token set

🌐 Starting ngrok tunnel...


INFO:     Started server process [226]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


✅ Public URL: https://unjestingly-tariffless-rozanne.ngrok-free.dev

📋 Copy this URL to your backend:
   VOICE_REMOTE_URL=https://unjestingly-tariffless-rozanne.ngrok-free.dev

🚀 Voice Service starting on https://unjestingly-tariffless-rozanne.ngrok-free.dev
   Local: http://127.0.0.1:8001
   Docs: https://unjestingly-tariffless-rozanne.ngrok-free.dev/docs

⏹️  Press stop button to halt the service
📝 Transcribing 106556 bytes with local Whisper...
✅ Transcribed:  A...
INFO:     2409:40e1:300a:10d0:7428:4766:9ba3:7366:0 - "POST /transcribe HTTP/1.1" 200 OK
📝 Transcribing 190598 bytes with local Whisper...
✅ Transcribed:  I have a headache from last 2 days....
INFO:     2409:40e1:300a:10d0:7428:4766:9ba3:7366:0 - "POST /transcribe HTTP/1.1" 200 OK
📝 Transcribing 89168 bytes with local Whisper...
✅ Transcribed:  I have a headache from last 2 days....
INFO:     2409:40e1:300a:10d0:7428:4766:9ba3:7366:0 - "POST /transcribe HTTP/1.1" 200 OK


## Testing Endpoints

Once the service is running, you can test it:

### Using curl:
```bash
# Test transcribe
curl -X POST {public_url}/transcribe -F "file=@audio.webm"

# Test synthesize
curl -X POST {public_url}/synthesize -H "Content-Type: application/json" -d '{"text": "Hello world"}'

# Test S2S
curl -X POST {public_url}/s2s -F "file=@audio.webm"
```

### Using Postman:
1. Import the Postman collection from your backend repo
2. Update the base URL to your ngrok public URL
3. Test each endpoint

### Using your frontend:
1. Set `VOICE_REMOTE_URL` to your ngrok URL
2. Frontend will automatically call the voice service
3. Test the voice button in chat

## Troubleshooting

### "ModuleNotFoundError: No module named 'TTS'" in Cell 4
- This is handled automatically - Cell 4 will install TTS if missing
- If still fails: Run Cell 1 again to ensure TTS installs
- Alternative: Manually run `!pip install --upgrade TTS` before Cell 4

### Session expires / URL changes
- Free ngrok URLs change on restart
- Set `NGROK_AUTH_TOKEN` in Cell 2 for permanent URLs
- Rerun cells 3-5 to get new URL

### Model download slow
- First TTS model load: 30-60s (downloads ~1.5GB)
- Subsequent calls: 5-15s (cached)
- Ensure Colab has GPU enabled (Runtime > Change runtime type > GPU)

### OpenAI API errors
- Verify API key is correct and has credit
- Check rate limits at https://platform.openai.com/account/rate-limits
- Whisper API is separate from chat API - verify you have audio usage

### Connection timeout
- Colab may have network issues
- Try: Rerun Cell 5 to restart service
- Use Colab Pro for more stable connections